In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

/opt/anaconda3/envs/investmentepfl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load Data
df = pd.read_csv('../arthur_hf_scraping/ms/ms_funds_returns.csv', index_col=0, parse_dates=True)

# Train/Test Split
train_start = '2020-01-01'
train_end = '2022-12-31'
test_start = '2023-01-01'
test_end = '2026-12-31'

print(f"- Number of funds in the dataset originally: {len(df.columns)}")
print(f"- Number of funds that have a full history during train and test: {df.loc[train_start:test_end].dropna(axis=1, how='any').shape[1]}")

df_train = df.loc[train_start:train_end].dropna(axis=1, how='any')
df_test = df.loc[test_start:test_end]


- Number of funds in the dataset originally: 675
- Number of funds that have a full history during train and test: 0


In [3]:
# Selection Criteria
vol = df_train.std() * np.sqrt(12)
ret = df_train.mean()

# Top 35% of volatility (quantile 0.65 means we want the highest 35%)
vol_threshold = vol.quantile(0.65)
high_vol_funds = vol[vol >= vol_threshold].index

# Filter returns for high vol funds
ret_filtered = ret.loc[high_vol_funds]

# Top 50% of returns among high vol funds
ret_threshold = ret_filtered.quantile(0.50)
final_funds = ret_filtered[ret_filtered >= ret_threshold].index.tolist()

print(f"- Number of funds selected as our 'top funds': {len(final_funds)}")

# Filter test set
test_data = df_test[final_funds].dropna(axis=1, how='any')
print(f"Test data shape: {test_data.shape}")


- Number of funds selected as our 'top funds': 72
Test data shape: (48, 0)


In [4]:
# Monte Carlo Heatmap for N=10
N = 10
n_simulations = 5000

def calculate_mdd(returns):
    cum_ret = (1 + returns).cumprod()
    running_max = cum_ret.cummax()
    drawdown = (cum_ret - running_max) / running_max
    return drawdown.min()

sim_returns = []
sim_vols = []
sim_mdds = []

np.random.seed(42)
available_funds = test_data.columns.tolist()

if len(available_funds) >= N:
    for _ in tqdm(range(n_simulations), desc='Simulating Portfolios (N=10)'):
        selected_funds = np.random.choice(available_funds, size=N, replace=False)
        portfolio_returns = test_data[selected_funds].mean(axis=1) # Equal Weight
        
        ann_ret = portfolio_returns.mean() * 12
        ann_vol = portfolio_returns.std() * np.sqrt(12)
        mdd = calculate_mdd(portfolio_returns)
        
        sim_returns.append(ann_ret)
        sim_vols.append(ann_vol)
        sim_mdds.append(mdd)

    sim_returns_arr = np.array(sim_returns)
    sim_vols_arr = np.array(sim_vols)
    sim_mdds_arr = np.array(sim_mdds)
    
    # --- Plot 1: Return vs Volatility ---
    ret_min, ret_max = np.min(sim_returns_arr), np.max(sim_returns_arr)
    vol_min, vol_max = np.min(sim_vols_arr), np.max(sim_vols_arr)
    
    x_grid = np.linspace(ret_min, ret_max, 100)
    y_grid_vol = np.linspace(vol_min, vol_max, 100)
    X_vol, Y_vol = np.meshgrid(x_grid, y_grid_vol)
    Z_vol = np.zeros_like(X_vol)
    
    for i in range(X_vol.shape[0]):
        for j in range(X_vol.shape[1]):
            x = X_vol[i, j]
            y = Y_vol[i, j]
            prop = np.sum((sim_returns_arr > x) & (sim_vols_arr > y)) / n_simulations
            Z_vol[i, j] = prop

    plt.figure(figsize=(10, 6))
    c1 = plt.contourf(X_vol, Y_vol, Z_vol, levels=50, cmap='viridis')
    plt.colorbar(c1, label='Proportion (Return > x & Vol > y)')
    
    cs1 = plt.contour(X_vol, Y_vol, Z_vol, levels=[0.05, 0.10, 0.25, 0.50], colors=['red', 'orange', 'yellow', 'white'], linewidths=2)
    plt.clabel(cs1, inline=True, fontsize=12, fmt='%.2f')

    plt.title(f'Monte Carlo Portfolios (N={N}) - Proportion(Ret > x & Vol > y)')
    plt.xlabel('Annualized Return (x)')
    plt.ylabel('Annualized Volatility (y)')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # --- Plot 2: Return vs MDD ---
    mdd_min, mdd_max = np.min(sim_mdds_arr), np.max(sim_mdds_arr)
    
    y_grid_mdd = np.linspace(mdd_min, mdd_max, 100)
    X_mdd, Y_mdd = np.meshgrid(x_grid, y_grid_mdd)
    Z_mdd = np.zeros_like(X_mdd)
    
    for i in range(X_mdd.shape[0]):
        for j in range(X_mdd.shape[1]):
            x = X_mdd[i, j]
            y = Y_mdd[i, j]
            prop = np.sum((sim_returns_arr > x) & (sim_mdds_arr > y)) / n_simulations
            Z_mdd[i, j] = prop

    plt.figure(figsize=(10, 6))
    c2 = plt.contourf(X_mdd, Y_mdd, Z_mdd, levels=50, cmap='viridis')
    plt.colorbar(c2, label='Proportion (Return > x & MDD > y)')
    
    cs2 = plt.contour(X_mdd, Y_mdd, Z_mdd, levels=[0.05, 0.10, 0.25, 0.50], colors=['red', 'orange', 'yellow', 'white'], linewidths=2)
    plt.clabel(cs2, inline=True, fontsize=12, fmt='%.2f')

    plt.title(f'Monte Carlo Portfolios (N={N}) - Proportion(Ret > x & MDD > y)')
    plt.xlabel('Annualized Return (x)')
    plt.ylabel('Maximum Drawdown (y)')
    plt.grid(True, alpha=0.3)
    plt.show()

else:
    print(f"Not enough funds to simulate N={N}.")


Not enough funds to simulate N=10.


In [5]:
# MDD and Calmar vs N
def calculate_cagr(returns):
    cum_ret = (1 + returns).prod()
    n_years = len(returns) / 12
    return (cum_ret ** (1 / n_years) - 1) if n_years > 0 else np.nan

N_values = [2, 4, 6, 8, 10, 15, 20, 25, 30]
n_sim_per_n = 1000

mdd_stats = {'N': [], 'median': [], 'q1': [], 'q3': []}
calmar_stats = {'N': [], 'median': [], 'q1': [], 'q3': []}

for n in tqdm(N_values, desc='Testing various N'):
    if n > len(available_funds):
        continue
    
    mdds = []
    calmars = []
    for _ in range(n_sim_per_n):
        selected_funds = np.random.choice(available_funds, size=n, replace=False)
        portfolio_returns = test_data[selected_funds].mean(axis=1)
        
        mdd = calculate_mdd(portfolio_returns)
        cagr = calculate_cagr(portfolio_returns)
        calmar = cagr / abs(mdd) if mdd != 0 else np.nan
        
        mdds.append(mdd)
        calmars.append(calmar)
        
    mdd_stats['N'].append(n)
    mdd_stats['median'].append(np.median(mdds))
    mdd_stats['q1'].append(np.percentile(mdds, 25))
    mdd_stats['q3'].append(np.percentile(mdds, 75))
    
    calmar_stats['N'].append(n)
    calmar_stats['median'].append(np.nanmedian(calmars))
    calmar_stats['q1'].append(np.nanpercentile(calmars, 25))
    calmar_stats['q3'].append(np.nanpercentile(calmars, 75))


Testing various N: 100%|██████████| 9/9 [00:00<00:00, 249991.63it/s]


In [6]:
# Plotting MDD vs N
if len(mdd_stats['N']) > 0:
    plt.figure(figsize=(10, 6))
    plt.plot(mdd_stats['N'], mdd_stats['median'], marker='o', label='Median MDD', color='red')
    plt.fill_between(mdd_stats['N'], mdd_stats['q1'], mdd_stats['q3'], color='red', alpha=0.2, label='Q1 - Q3 Range')
    plt.title('Max Drawdown vs Portfolio Size (N)')
    plt.xlabel('N (Number of Funds)')
    plt.ylabel('Max Drawdown')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # Plotting Calmar Ratio vs N
    plt.figure(figsize=(10, 6))
    plt.plot(calmar_stats['N'], calmar_stats['median'], marker='o', label='Median Calmar', color='blue')
    plt.fill_between(calmar_stats['N'], calmar_stats['q1'], calmar_stats['q3'], color='blue', alpha=0.2, label='Q1 - Q3 Range')
    plt.title('Calmar Ratio vs Portfolio Size (N)')
    plt.xlabel('N (Number of Funds)')
    plt.ylabel('Calmar Ratio')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("Not enough funds for testing N_values.")


Not enough funds for testing N_values.
